# 06 — Serving bundle export

Export the supplied Referee checkpoint as a hashed `safetensors` serving bundle. This is packaging, not model evaluation; run notebook 04 for validation metrics first.


## Setup

```sh
uv sync --project models/referee --frozen
```


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import hashlib
import json
import os

import pandas as pd
from IPython.display import display

from referee_evaluation import notebook_tools

REPOSITORY_ROOT = Path.cwd().resolve()
while not (REPOSITORY_ROOT / 'models' / 'referee' / 'training').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
MODEL_PROJECT = REPOSITORY_ROOT / 'models' / 'referee'


## Options

`rgb_resize` is provisional: it matches the current `training/dataset.py` transform, but the checkpoint has no preprocessing provenance. Confirm with notebook 04 before enabling the Referee in production.


In [ ]:
RUN_EXPORT = True
CHECKPOINT = MODEL_PROJECT / 'checkpoints' / 'referee-epoch=04-val_mean_ap=0.5564.ckpt'
MODEL_ID = 'referee-vit-a-v1'
PREPROCESSING = 'rgb_resize'  # or foreground after provenance review
BUNDLE_ROOT = MODEL_PROJECT / 'checkpoints' / 'referee-evaluation' / 'bundles'
OVERWRITE = False

print('Export enabled:', RUN_EXPORT)
print('Checkpoint:', CHECKPOINT)
print('Bundle:', BUNDLE_ROOT / MODEL_ID)
print('Preprocessing:', PREPROCESSING)


## Readiness


In [ ]:
print('Checkpoint present:', CHECKPOINT.is_file())
print('Exporter present:', (MODEL_PROJECT / 'training' / 'export_bundle.py').is_file())
if not CHECKPOINT.is_file():
    raise FileNotFoundError('Place the supplied Referee checkpoint under models/referee/checkpoints/ first.')


## Export

The exporter writes a temporary bundle, verifies every file hash and fixed-tensor parity, then publishes atomically. Existing bundles are left untouched unless `OVERWRITE = True`.


In [ ]:
MODEL_ENV = dict(os.environ)
MODEL_ENV.pop('PYTHONPATH', None)
MODEL_ENV.pop('PYTHONHOME', None)

if not RUN_EXPORT:
    print('Export disabled.')
else:
    command = [
        'uv', 'run', '--project', str(MODEL_PROJECT), '--frozen', '--no-sync',
        'python', '-m', 'training.export_bundle',
        '--checkpoint', str(CHECKPOINT),
        '--model-id', MODEL_ID,
        '--preprocessing', PREPROCESSING,
        '--output-dir', str(BUNDLE_ROOT),
    ]
    if OVERWRITE:
        command.append('--overwrite')
    print('Launching:', ' '.join(command))
    if not notebook_tools.run_interruptible_command(command, cwd=MODEL_PROJECT, env=MODEL_ENV):
        raise RuntimeError('Export interrupted.')


## Verify bundle

Re-read the manifest, hash every file and confirm the four-label order. This does not load the model.


In [ ]:
BUNDLE = BUNDLE_ROOT / MODEL_ID
if not (BUNDLE / 'manifest.json').is_file():
    print('No bundle yet.')
else:
    manifest = json.loads((BUNDLE / 'manifest.json').read_text(encoding='utf-8'))
    rows = []
    for name, expected in manifest['files'].items():
        actual = hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest()
        rows.append({'file': name, 'hash_match': actual == expected, 'bytes': (BUNDLE / name).stat().st_size})
    display(pd.DataFrame(rows).set_index('file'))
    print('modelId:', manifest['modelId'])
    print('revision:', manifest['artifactRevision'])
    print('preprocessing:', manifest['preprocessing'])
    print('labels:', (BUNDLE / 'labels.txt').read_text(encoding='utf-8').splitlines())
    print('channel order:', manifest['channelOrder'])


## Integration note

Copy the bundle to the AI artifact root as `<AI_ARTIFACT_ROOT>/<modelId>/` and configure the service with `REFEREE_MODEL_ID=referee-vit-a-v1`, `REFEREE_PROVIDER=http`, `AI_TEST_FIXTURES=0`. The Referee remains an advisory feature; enabling it in a public deployment still requires validation and review.
